# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI


In [ ]:
# constants

MODEL_GEMINI = 'gemini-3.1-flash-lite'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")
base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=api_key,base_url=base_url)
llama = OpenAI(base_url="http://localhost:11434/v1",
               api_key="ollama")

In [ ]:
# here is the question; type over this to ask something new

def greeting():
    print("""
    =========================================
              WELCOME TO DSA MENTOR
    =========================================

    Hi! I am your personal DSA mentor.

    I can help you understand and solve
    Data Structures and Algorithms problems.

    I will explain the approach, write code
    in your preferred language, and walk you
    through an example step by step.
    """)

In [ ]:
# Get gpt-4o-mini to answer, with streaming

def solve_problem(problem, language, level, questions):

    # System prompt - DSA tutor
    system_prompt = """
    You are an expert Data Structures and Algorithms tutor.

    Your job is to help students understand and solve DSA problems.

    Follow these instructions:

    1. Explain the problem in very simple English.
    2. Explain the brute force approach first.
    3. Explain the optimized approach.
    4. Identify the data structures and algorithms used.
    5. Write the complete solution in the programming language
       requested by the user.
    6. Explain the code line by line.
    7. Perform a detailed dry run using an easy example.
    8. Show how variables change at each step.
    9. Explain time and space complexity in simple terms.
    10. Discuss important edge cases.

    Make your explanation beginner-friendly.
    Do not skip steps or assume prior knowledge.

    Respond in Markdown only.
    Do not wrap the entire response in a code block.
    """

    user_prompt = """
    Solve the following DSA problem and teach me how to
    understand and approach it.

    Problem:
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                user_prompt
                + problem
                + "\nProgramming language: " + language
                + "\nMy current level: " + level
                + "\nAdditional questions: " + questions
            )
        }
    ]

    # OpenAI response
    print("\n\n========== PART 1: OPENAI SOLUTION ==========\n")

    openai_response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=messages,
        stream=True
    )

    for chunk in openai_response:
        content = chunk.choices[0].delta.content
        if content:
            print(content,end ="",flush=True)

    # Llama response
    print("\n\n========== PART 2: LLAMA SOLUTION ==========\n")
 
    llama_response = llama.chat.completions.create(
        model="llama3.2:latest",
        messages=messages,
        stream=True
    )


    for chunk in llama_response:
        content = chunk.choices[0].delta.content

        if content:
            print(content, end="", flush=True)


In [ ]:
def get_language(prompt):
    valid_lang = ["python","java","c++"]
    while True:
        language = input(prompt).strip().lower()
        if language in valid_lang:
            return language
        print("Invalid language. Choose Python, Java, C++.")



In [ ]:
def get_level(prompt):

    valid_levels = ["beginner", "intermediate", "advanced"]

    while True:
        level = input(prompt).strip().lower()

        if level in valid_levels:
            return level

        print("Please choose beginner, intermediate or advanced.")

In [ ]:
def get_string(prompt):
    return input(prompt).strip()

In [ ]:
def get_user_data():

    problem = get_string("\nPaste your DSA problem:\n")

    language = get_language(
        "\nWhich programming language do you prefer? (Python/Java/C++/JavaScript): "
    )

    level = get_level(
        "\nWhat is your current level? (Beginner/Intermediate/Advanced): "
    )

    questions = get_string(
        "\nAny specific questions or concepts you want explained? "
    )

    solve_problem(
        problem=problem,
        language=language,
        level=level,
        questions=questions
    )

In [ ]:
greeting()
get_user_data()